In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch transformers datasets peft bitsandbytes py_vncorenlp \
    accelerate trl \
    numpy pandas scikit-learn sentencepiece tokenizers tqdm underthesea

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.7 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=8d59d4a257a80045348f062bbb80a98d0bd14affc541c3989ebe6f677ac98303
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [3]:
!git clone https://github.com/Tommachilez/improving-learned-index.git
%cd /content/improving-learned-index

Cloning into 'improving-learned-index'...
remote: Enumerating objects: 1235, done.
remote: Counting objects: 100% (331/331), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 1235 (delta 291), reused 282 (delta 264), pack-reused 904 (from 1)
Receiving objects: 100% (1235/1235), 203.05 KiB | 10.15 MiB/s, done.
Resolving deltas: 100% (903/903), done.
/content/improving-learned-index


# Indexing

In [4]:
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
doc_mapping = f"{dataset_path}/unique_doc_mapping.csv"
pretokenized_queries = f"{dataset_path}/vn_mining/queries_pretokenized.jsonl"
expanded_passages = f"{dataset_path}/deeperimpact/passage_dir"
passage_tsv = f"{expanded_passages}/passages.tsv"
output_collection_path = f"{dataset_path}/deeperimpact/collection.index"
quantized_collection_path = f"{dataset_path}/deeperimpact/collection.index.quantized"
inverted_indices_path = f"{dataset_path}/deeperimpact/inverted_indices"

query_mapping = f"{dataset_path}/test_query_mapping.csv"
test_vifc = f"{dataset_path}/vifc_test_set.csv"
test_queries = f"{dataset_path}/deeperimpact/test_queries.tsv"
test_qrels = f"{dataset_path}/deeperimpact/test_qrels.tsv"

In [5]:
!python -m src.deep_impact.scripts.create_passages \
    --input_csv {doc_mapping} \
    --queries_jsonl {pretokenized_queries} \
    --output_dir {expanded_passages}

Loading expansion terms from /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining/queries_pretokenized.jsonl with max_terms=100...
Parsing queries: 61931it [00:01, 36866.65it/s]
Loaded expansion terms for 3030 documents.
Processing documents from /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv...
Creating passages: 3030it [00:00, 5700.70it/s]
Done. Processed 10027 total passages.
Outputs saved to:
  - Passages: /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/passage_dir/passages.tsv
  - Mapping:  /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/passage_dir/pid_mapping.txt


In [6]:
!python -m src.deep_impact.index \
  --collection_path {passage_tsv} \
  --output_file_path {output_collection_path} \
  --model_checkpoint_path /content/drive/MyDrive/KLTN_Project/Models/deeperimpact_xlmr_revised/DeepImpact_latest.pt \
  --model_batch_size 32 \
#   --process_batch_size <process_batch_size> \
#   --num_processes <n> \

2025-12-26 14:46:57.471689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766760417.491156    1324 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766760417.497088    1324 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766760417.512114    1324 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766760417.512140    1324 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766760417.512145    1324 computation_placer.cc:177] computation placer alr

In [7]:
!python -m src.deep_impact.indexing.quantize \
  -i {output_collection_path} \
  -o {quantized_collection_path}

10027it [00:01, 6423.87it/s]
Found max value: 1.2589999437332153
10027it [00:02, 4826.50it/s]


In [8]:
!python -m src.deep_impact.inverted_index.create \
  -i {quantized_collection_path} \
  -o {inverted_indices_path}

100% 10027/10027 [00:01<00:00, 6771.24it/s]
100% 50199/50199 [00:00<00:00, 708514.18it/s]
100% 10027/10027 [00:01<00:00, 6940.35it/s]
50199it [00:02, 24182.23it/s]
100% 50199/50199 [00:00<00:00, 1304675.68it/s]


# Ranking

In [5]:
!python -m src.deep_impact.scripts.create_test_files \
    --test_query_mapping {query_mapping} \
    --vifc_file {test_vifc} \
    --doc_mapping {doc_mapping} \
    --output_queries {test_queries} \
    --output_qrels {test_qrels}

Loading document mapping from /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv...
Loading Docs: 3030it [00:00, 3543.17it/s]
Loading VIFC data from /content/drive/MyDrive/KLTN_Project/Datasets/vifc_test_set.csv...
Loading VIFC: 7687it [00:02, 2963.41it/s]
Processing queries and generating outputs...
Processing: 7685it [00:01, 7067.90it/s]

Processing Complete.
Generated 7685 queries in /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/test_queries.tsv
Generated 7686 qrels in /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/test_qrels.tsv


In [6]:
!python -m src.deep_impact.rank \
  --index_path {inverted_indices_path} \
  --queries_path {test_queries} \
  --qrels_path {test_qrels} \
  --output_path /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/raw_run_file.txt \
  --dataset_type msmarco \
#   --num_workers <n>

2025-12-26 16:04:35.279363: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766765075.301494    2183 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766765075.308298    2183 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766765075.323334    2183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766765075.323359    2183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766765075.323363    2183 computation_placer.cc:177] computation placer alr

In [8]:
!python -m src.deep_impact.aggregate_run \
    --run_file /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/raw_run_file.txt \
    --mapping /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/passage_dir/pid_mapping.txt \
    --output /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/run_file.txt

Loading ID mapping...
Aggregating results...
7685000it [00:10, 741206.36it/s]
Writing to /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/run_file.txt...


In [9]:
!python -m src.deep_impact.evaluate \
  --run_file_path /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/run_file.txt \
  --qrels_path {test_qrels}

2025-12-26 16:26:05.022484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766766365.042965    7573 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766766365.048989    7573 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766766365.065092    7573 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766766365.065118    7573 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766766365.065126    7573 computation_placer.cc:177] computation placer alr